In [2]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import loadtxt
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import SGDClassifier
from lightgbm import LGBMClassifier


In [3]:
dataset_druggable = pd.read_csv("input/combined_DepMap_21Q3_druggable.csv")
dataset_ccle = pd.read_csv("input/combined_DepMap_21Q3_CCLE_expression.csv")
druggable_X = dataset_druggable.iloc[:, 1:-4686]
druggable_y = dataset_druggable.iloc[:, -4686:]
ccle_X = dataset_ccle.iloc[:, 1:-4686]
ccle = dataset_ccle.iloc[:, -4686:]

In [4]:
dataset = pd.read_csv("input/combined_DepMap_21Q3.csv")
num_gene = 17651
drug_X = dataset.iloc[:, 1:num_gene+1]
drug_y = dataset.iloc[:, -4686:]
drug_list = drug_y.columns.tolist()

In [5]:
# SOME PARAMETERS FOR GRIDSEARCH
GRID_SEARCH_PARAM = {
'XGB': {
		"n_estimators": [100, 150],
 },
 'RF':{
	 "n_estimators": [100, 150, 250],
	 "max_depth" : [20, 50, 100, 200],
 },
 'SGD':{
	 'l1_ratio':[0.2, 0.15, 0.1, 0.05],
	 'alpha':[0.02, 0.05]
 }
}

# Basic function to handel sklearn and traditional model training and basic hyperparameter optimization
# TODO refactor this into a class maybe, class DrugModel
def skl_drug_model(X, Y, drug, model = XGBClassifier, oversample = True, fixed_params={}, search_params = {}, search_method = 'gridcv'):

	assert model in [XGBClassifier, RandomForestClassifier, SGDClassifier, LGBMClassifier], f' {model} type not supported'

	
	y = Y[drug]

	 # split X and y into training and testing sets
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)


	if oversample:
		ros = SMOTE(random_state=72)
		X_res, y_res = ros.fit_resample(X_train, y_train)
		oversample = 'oversample'
	else:
		X_res, y_res = X_train, y_train
		oversample = ''


	 # instantiate the classifier
	 

	# k-fold cross validation using multiple metric evaluation
	kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
	 #cv_results = cross_validate(xgbc, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1'], n_jobs = 10)
	if oversample:
		imba_pipeline = Pipeline([('sampling', SMOTE(random_state=72)), 
							  ('classifier', model(n_jobs=20, **fixed_params))])
		grid_search_parameters = {'classifier__' + key: search_params[key] for key in search_params}
	else:
		imba_pipeline = model(**fixed_params, n_jobs=20)
	 #cross_val_score(imba_pipeline, X_train, y_train, scoring='recall', cv=kf)
	model0 = model(**fixed_params, n_jobs=20)

	 #perform gridsearch if needed
	if search_method == 'gridcv' and search_params:
		grid_imba = GridSearchCV(imba_pipeline, param_grid=grid_search_parameters, cv=kfold, scoring='precision',
						return_train_score=True)
		grid_imba.fit(X_train, y_train) 

		best_params = {key.removeprefix('classifier__'):grid_imba.best_params_[key] for key in grid_imba.best_params_}
		print(best_params)
		model0.set_params(**best_params)
	else:
		grid_imba = None
	 
	if oversample:
		imba_pipeline = Pipeline([('sampling', RandomOverSampler(random_state=72)), 
							  ('classifier', model0)])
	else:
		imba_pipeline = model0
		  
	# generate cross validation results of best model with correct oversampling 
	# OVERSAMPLING must come after validation split for correct validation , thus the use of pipeline
	# cross_validate function will first split into train/validate, then feed training data into pipeline (oversampling + training)
	cv_results = cross_validate(imba_pipeline, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs = 20)
	cv_results = pd.DataFrame(cv_results)
	

	 # declare parameters
	 

	 # fit the classifier to the training data
	model0.fit(X_res, y_res)

	 # save the trained model
	
		  
	final_results = {
		'best_model': model0,
		'drug': drug,
		'model_class': model,
		'model_search': grid_imba,
		'search_method': search_method,
		'X_test': X_test,
		'Y_test': y_test,
		'X_train': X_train,
		'Y_train': y_train,
		'cv_results':cv_results,
		'oversample': True if oversample else False
	}

	return final_results


def write_drug_model_result(model_results, out_dir):
	 
	model0 = model_results['best_model']
	oversample = 'oversample' if model_results['oversample'] else ''
	drug = model_results['drug']
	model_class = model_results['model_class']

	model_name = {
		  XGBClassifier:'XGBClassifier',
		  RandomForestClassifier:'RandomForestClassifier',
		  SGDClassifier:'SGDClassifier',
		  LGBMClassifier:'LGBMClassifier'

	}[model_class]

	full_out_dir = f'{out_dir}/{oversample}/{drug}/'
	X_test, y_test = model_results['X_test'], model_results['Y_test']
	X_train, y_train = model_results['X_train'], model_results['Y_train']
	y_pred = model0.predict(X_test)

	os.makedirs(full_out_dir, exist_ok=True)

	joblib.dump(model0, f'{full_out_dir}/{model_name}_{drug}.joblib')
	
	model_results['cv_results'].to_csv(f'{full_out_dir}/{model_name}_cv_results_{drug}.csv')

	print(drug,
		 f'{model_class}_model_parameters', model0, "\n",
		 "confusion_matrix:", "\n", confusion_matrix(y_test, y_pred), "\n",
		 file=open(f'{full_out_dir}/{model_name}_confusion_matrix.txt', "a"))

	model_report = classification_report(y_test, y_pred, output_dict=True, labels=np.unique(y_pred))
	model_report = pd.DataFrame(model_report).transpose()
	
	if (model_report.index == "1").any() == True:
		r1 = pd.DataFrame(model_report.loc["1"]).transpose()
		r1.to_csv(f'{full_out_dir}/{model_name}_classification_report_{drug}.csv')
	else:
		print(drug, "Nothing predicted as 1",
			   file=open(f'{out_dir}/{oversample}/{model_name}_classification_report_log.txt', "a"))

	 

	 # feature importance with XGBoost
	if model_class in [XGBClassifier, RandomForestClassifier]:
		fi = pd.DataFrame({'feature': list(X_train.columns),
					'importances': model0.feature_importances_ * 100}).\
					 sort_values('importances', ascending = False)
		fi.to_csv(f'{full_out_dir}/{model_name}_feature_importance_{drug}.csv')

In [8]:
model_map = {}
for d in drug_list[0:10]:
	model_map[d] = skl_drug_model(drug_X, drug_y, d, 
		model = SGDClassifier, 
		oversample = True, 
		fixed_params={'loss':'modified_huber', 'penalty':'elasticnet', 'learning_rate':'optimal'},
		search_params=GRID_SEARCH_PARAM['SGD']
		)

{'alpha': 0.02, 'l1_ratio': 0.1}
{'alpha': 0.02, 'l1_ratio': 0.15}
{'alpha': 0.02, 'l1_ratio': 0.05}
{'alpha': 0.02, 'l1_ratio': 0.05}
{'alpha': 0.02, 'l1_ratio': 0.05}
{'alpha': 0.02, 'l1_ratio': 0.05}
{'alpha': 0.02, 'l1_ratio': 0.05}
{'alpha': 0.02, 'l1_ratio': 0.05}
{'alpha': 0.02, 'l1_ratio': 0.1}
{'alpha': 0.05, 'l1_ratio': 0.05}


In [ ]:
model_map = {}
for d in drug_list[0:1]:
	model_map[d] = skl_drug_model(drug_X, drug_y, d, 
		model = LGBMClassifier, 
		oversample = False,
		search_params={},
		fixed_params={}
		)

[LightGBM] [Info] Number of positive: 205, number of negative: 534
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.201360 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4354814
[LightGBM] [Info] Number of data points in the train set: 739, number of used features: 17651
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.277402 -> initscore=-0.957386
[LightGBM] [Info] Start training from score -0.957386
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

In [ ]:
write_drug_model_result(model_map['BRD-A00077618-236-07-6::2.5::HTS'], out_dir = 'output')

In [116]:
model_map['BRD-A00077618-236-07-6::2.5::HTS']['cv_results']

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,66.024491,3.063161,0.716216,0.444444,0.097561,0.160000,0.718031
1,65.446071,3.108201,0.797297,0.923077,0.292683,0.444444,0.816047
2,66.254138,3.074916,0.770270,0.705882,0.292683,0.413793,0.811716
3,64.390126,3.062809,0.770270,0.888889,0.195122,0.320000,0.808525
4,65.176987,3.067385,0.748299,0.642857,0.219512,0.327273,0.709848


In [24]:
starttime = time.time()
grid_search = skl_drug_model('BRD-A00100033-001-08-9::2.5::HTS',   oversample = True, model = RandomForestClassifier
)
endtime = time.time()

{'max_depth': 200, 'n_estimators': 100}
